# AAI-540 Group 2 — 30-Day Hospital Readmission Risk

**School:** University of San Diego, CA  
**Authors:** Jose Sandoval, Manikanta Katuri, Michael Domingo  
**Course:** AAI-540 — Machine Learning Operations

**Project summary:** Predict 30-day hospital readmission risk on CMS DE-SynPUF Medicare claims to help care teams target post-discharge interventions.  
The notebook trains a logistic-regression baseline and an XGBoost model with a patient-grouped 40 / 30 / 30 split, gated on AUC ≥ 0.75 before any deployment.  
End-to-end MLOps is delivered on Amazon SageMaker — Feature Store, Model Registry, real-time endpoint, Model Monitor drift checks, and CloudWatch alarms + dashboard — driven by GitHub Actions CI/CD.

What it covers, end to end:
1. Configuration
2. Data ingest (CMS DE-SynPUF from S3, or deterministic synthetic generator for development)
3. 30-day readmission labeling + prior-utilization features
4. **40 / 30 / 30** train / test / validation split (patient-grouped, time-aware) — per AAI-540 final-project requirement
5. EDA + class-imbalance check
   - 5b. **Feature engineering** — winsorise, bucket, encode (same code path as training)
   - 5c. **Feature Store** — register the curated features (offline + online stores)
   - 5d. **Offline-store via Athena** — training-set assembly + cohort SQL against the Glue-cataloged feature table
6. Baseline (logistic regression) + primary (XGBoost) training
7. Subgroup / fairness evaluation
8. Final test-set evaluation + AUC deploy gate
   - 8b. **Model Store** — package and register the trained model in the Model Registry
9. **SageMaker Pipelines** orchestration (Preprocess → Train → Evaluate → AUC gate → Register)
10. Real-time endpoint deployment
11. **Model Monitor** baselines + hourly schedule
12. **CloudWatch alarms + dashboard** (no email subscribers — alerts and metrics are reviewed in the AWS console)
13. Live inference test
14. Cleanup

All code under `src/readmit/` is the same code that CI/CD runs in production — this notebook just drives it.


## 0. Environment setup

Some SageMaker Studio kernels ship only `sagemaker-core` (the modular v3 SDK), which owns the `sagemaker` namespace but is missing `sagemaker.session`, `sagemaker.feature_store`, `sagemaker.get_execution_role`, etc. — everything this project uses.

The cell below checks for the classic v2 SDK and installs `requirements.txt` only if needed, so **kernel restart + run-all** is fully self-contained. It is idempotent: on a kernel that already has the right SDK it just prints the version and skips pip.

In [ ]:
# Idempotent: only installs when the classic SageMaker SDK (v2) is missing or shadowed.
import os, sys, subprocess

def _classic_sagemaker_ok() -> bool:
    """True only if the classic v2 SageMaker Python SDK is importable."""
    try:
        import sagemaker  # noqa: F401
        from sagemaker.session import Session  # noqa: F401  -- absent in v3/core
        ver = getattr(sagemaker, "__version__", "")
        return ver.startswith("2.")
    except Exception:
        return False

def _find_requirements_txt() -> str | None:
    """Walk upward from the notebook's directory looking for requirements.txt."""
    start = globals().get('__vsc_ipynb_file__') or globals().get('__file__') or os.getcwd()
    d = os.path.abspath(os.path.dirname(start) if os.path.isfile(start) else start)
    for _ in range(6):
        cand = os.path.join(d, 'requirements.txt')
        if os.path.isfile(cand):
            return cand
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None

if _classic_sagemaker_ok():
    import sagemaker
    print(f"sagemaker {sagemaker.__version__} already present — skipping install.")
else:
    print("Classic SageMaker SDK not found on this kernel — installing...")
    req = _find_requirements_txt()
    if req:
        print(f"  pip install -r {req}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", req])
    else:
        print("  requirements.txt not found — installing minimum pins only")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "sagemaker>=2.220,<3.0", "s3fs>=2024.3.0",
        ])
    # Drop any partially-loaded modules so subsequent imports pick up v2 cleanly.
    for _m in [k for k in list(sys.modules) if k == "sagemaker" or k.startswith("sagemaker.")]:
        del sys.modules[_m]
    import sagemaker
    print(f"Installed sagemaker {sagemaker.__version__}")

# Final assertion — fail loud here rather than 30 cells later in 5c.
from sagemaker.session import Session as _Session  # noqa: F401
from sagemaker.feature_store.feature_group import FeatureGroup as _FG  # noqa: F401
print("Classic SageMaker SDK surface OK (session + feature_store importable).")

## 1. Configuration

In [ ]:
# AWS region, the SageMaker execution role, and the default S3 bucket are NOT
# hardcoded here — they are resolved from the running session in the next cell
# so the same notebook works in any account / region.

# ---------------------------------------------------------------------------
# Make src/readmit importable even when the package hasn't been pip-installed.
# SageMaker Studio kernels don't always start in the notebook's directory, so
# walk upward from cwd (and a few common roots) until we find a `src/readmit`
# folder, then prepend its parent to sys.path.
# ---------------------------------------------------------------------------
import os, sys
def _bootstrap_readmit_path():
    candidates = [os.getcwd()]
    # Also try the directory the notebook lives in, if Jupyter exposes it.
    nb_dir = globals().get('__vsc_ipynb_file__') or globals().get('__file__')
    if nb_dir:
        candidates.append(os.path.dirname(os.path.abspath(nb_dir)))
    seen = set()
    for start in candidates:
        d = os.path.abspath(start)
        for _ in range(6):                       # walk up to 6 levels
            if d in seen:
                break
            seen.add(d)
            src = os.path.join(d, 'src')
            if os.path.isdir(os.path.join(src, 'readmit')):
                if src not in sys.path:
                    sys.path.insert(0, src)
                return src
            parent = os.path.dirname(d)
            if parent == d:
                break
            d = parent
    return None

_resolved = _bootstrap_readmit_path()
if _resolved is None:
    raise RuntimeError(
        "Could not locate src/readmit on disk. Run `pip install -e .` from the "
        "project root, or set PROJECT_ROOT explicitly and re-run this cell."
    )
print(f'readmit package path: {_resolved}')

# All three of these are resolved from the live SageMaker / STS session in the
# next cell. Leave as None unless you want to override (e.g. cross-account run).
BUCKET                  = None
SAGEMAKER_ROLE_ARN      = None
PROJECT_PREFIX          = 'readmit'

# Data source. Default is 'cms-open' which curates the public AWS Open Data
# CMS DE-SynPUF dataset (s3://synpuf-omop/, OMOP CDM v5.x) into
# s3://{BUCKET}/{PROJECT_PREFIX}/curated/encounters.parquet and reuses that
# parquet for every downstream cell. Other options:
#   'synthetic' -> deterministic in-memory generator (no S3 needed; what CI uses)
#   's3'        -> read a pre-curated encounters.parquet from S3_CURATED_URI
DATA_SOURCE             = 'cms-open'
CMS_TIER                = '100k'         # '1k' (CI / smoke) or '100k' (full demo)
S3_CURATED_URI          = None           # None -> auto-derive from BUCKET + PROJECT_PREFIX in the next cell
N_PATIENTS              = 50_000         # cap on patients returned (None = all)
FORCE_RECURATE          = False          # set True to ignore the cached encounters.parquet and rebuild from raw OMOP

# Endpoint + monitoring (no external subscribers — alerts/dashboards live in the AWS console)
ENDPOINT_NAME           = 'readmit-risk-dev'

# Feature Store + Model Registry
FEATURE_GROUP_NAME      = 'readmit-encounter-features'
MODEL_PACKAGE_GROUP     = 'ReadmitRiskModels'

# Required split (40 / 30 / 30) — locked in src/readmit/config.py
from readmit.config import TRAIN_FRAC, TEST_FRAC, VAL_FRAC
print(f'Split locked at train={TRAIN_FRAC} test={TEST_FRAC} val={VAL_FRAC}')


In [ ]:
import json, logging, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# --------------------------------------------------------------------------
# (Optional first-run install)
# The Configuration cell above already resolved the project root into the
# `_resolved` variable (it ends in `.../src`). Derive the project root from
# that so the install works regardless of where the kernel's cwd is — Studio
# kernels start in /home/sagemaker-user, not in /notebooks/.
#
# Uncomment the next block on a fresh kernel, run it once, then re-comment.
# --------------------------------------------------------------------------
# PROJECT_ROOT = os.path.abspath(os.path.join(_resolved, os.pardir))
# !pip install -q -r {PROJECT_ROOT}/requirements.txt && pip install -q -e {PROJECT_ROOT}

# Safety net: re-assert src/ on sys.path in case this cell is run on its own.
# The Configuration cell normally already did this via _bootstrap_readmit_path().
if '_resolved' in globals() and _resolved and _resolved not in sys.path:
    sys.path.insert(0, _resolved)

from readmit.config import FEATURES, LABEL_COL, AUC_THRESHOLD_DEPLOY
from readmit.data.ingest   import load_encounters
from readmit.data.labeling import attach_labels_and_priors
from readmit.data.splits   import split_train_test_val
from readmit.features.engineering import (
    add_derived_columns, build_preprocessor, winsorize_utilization
)
from readmit.models.evaluate import compute_metrics, recall_at_k, subgroup_metrics
from readmit.models.train    import build_xgb_pipeline, build_logreg_pipeline

# --------------------------------------------------------------------------
# Resolve session / region / role / bucket from the live AWS environment.
# Anything pre-set in the Configuration cell is preserved; everything else is
# derived from sagemaker.Session() + STS so the notebook is portable across
# accounts and regions without code changes.
# --------------------------------------------------------------------------
import boto3
import sagemaker

# ---- 1. SageMaker Session + region ----
try:
    sess = sagemaker.Session()
    AWS_REGION = sess.boto_region_name
except Exception as exc:
    logging.warning('sagemaker.Session() failed (%s); falling back to boto3.', exc)
    sess       = None
    AWS_REGION = boto3.session.Session().region_name or 'us-east-1'

# ---- 2. Account id via STS (works in any AWS context: Studio, Notebook
#         Instance, generic JupyterLab, EC2, ECS, local with credentials) ----
_sts = boto3.client('sts', region_name=AWS_REGION)
_caller = _sts.get_caller_identity()
ACCOUNT_ID = _caller['Account']
_caller_arn = _caller['Arn']                  # e.g. arn:aws:sts::<acct>:assumed-role/<RoleName>/<session>
logging.info('STS caller: %s', _caller_arn)

# ---- 3. Execution role -----------------------------------------------------
# Preferred path: sagemaker.get_execution_role() — works in real SageMaker kernels.
# Fallback: derive the underlying IAM role ARN from the assumed-role STS ARN.
# This works on Studio Classic, generic JupyterLab, EC2 with instance profile,
# anywhere `get_caller_identity` returns an assumed-role principal.
if SAGEMAKER_ROLE_ARN is None:
    try:
        SAGEMAKER_ROLE_ARN = sagemaker.get_execution_role()
    except Exception as exc:
        logging.info(
            'sagemaker.get_execution_role() unavailable (%s); '
            'deriving role from STS assumed-role principal.', exc,
        )
        # arn:aws:sts::ACCOUNT:assumed-role/ROLE_NAME/SESSION -> arn:aws:iam::ACCOUNT:role/ROLE_NAME
        if ':assumed-role/' in _caller_arn:
            role_name = _caller_arn.split(':assumed-role/', 1)[1].split('/', 1)[0]
            SAGEMAKER_ROLE_ARN = f'arn:aws:iam::{ACCOUNT_ID}:role/{role_name}'
        elif ':role/' in _caller_arn:
            SAGEMAKER_ROLE_ARN = _caller_arn       # already a plain role ARN
        else:
            SAGEMAKER_ROLE_ARN = os.environ.get('SAGEMAKER_ROLE_ARN')

# ---- 4. Default S3 bucket --------------------------------------------------
# Preferred path: sess.default_bucket() — creates sagemaker-<region>-<acct> if needed.
# Fallback: construct the canonical name ourselves from STS + region.
if BUCKET is None:
    if sess is not None:
        try:
            BUCKET = sess.default_bucket()
        except Exception as exc:
            logging.warning('sess.default_bucket() failed (%s); deriving from STS.', exc)
    if BUCKET is None:
        BUCKET = f'sagemaker-{AWS_REGION}-{ACCOUNT_ID}'

# ---- 5. Fail-fast verification BEFORE any S3 / SageMaker write -------------
_s3 = boto3.client('s3', region_name=AWS_REGION)
try:
    _s3.head_bucket(Bucket=BUCKET)
except Exception as exc:
    raise RuntimeError(
        f"BUCKET={BUCKET!r} is not reachable from this account/region ({exc}). "
        "Check that the SageMaker default bucket exists, or override BUCKET in "
        "the Configuration cell."
    )
if SAGEMAKER_ROLE_ARN is None:
    logging.warning(
        'SAGEMAKER_ROLE_ARN could not be resolved — Feature Store, Model '
        'Registry, Pipelines, Endpoint, Monitor and Alerts cells will skip.'
    )

# Derive the curated parquet location from the bucket once the session is up,
# so the rest of the notebook only ever talks to one S3 path.
if S3_CURATED_URI is None and DATA_SOURCE in ('cms-open', 's3'):
    S3_CURATED_URI = f's3://{BUCKET}/{PROJECT_PREFIX}/curated/'

print(f'AWS account       : {ACCOUNT_ID}')
print(f'AWS region        : {AWS_REGION}')
print(f'Execution role    : {SAGEMAKER_ROLE_ARN or "(not resolved — AWS cells will skip)"}')
print(f'Default S3 bucket : {BUCKET}')
print(f'Curated data URI  : {S3_CURATED_URI}')


## 2. Ingest encounters

In [ ]:
# First-call behaviour for source='cms-open':
#   1. Read OMOP tables (person, visit_occurrence, condition_occurrence) from
#      the public bucket s3://synpuf-omop/  (anonymous, us-east-1).
#   2. Join + derive the 14 columns the rest of the pipeline expects.
#   3. Write the curated frame to S3_CURATED_URI + 'encounters.parquet'.
#   4. Return it.
# Subsequent calls just read the cached parquet from our bucket — fast.

encounters = load_encounters(
    source=DATA_SOURCE,
    s3_uri=S3_CURATED_URI,
    n_patients=N_PATIENTS,
    seed=42,
    cms_tier=CMS_TIER,
    force_recurate=FORCE_RECURATE, # set to True to force re-curation, False to use cached data
)

print(f'Loaded {len(encounters):,} encounters across {encounters.beneficiary_id.nunique():,} patients')
print(f'Source           : {DATA_SOURCE}'
      + (f'  |  tier: {CMS_TIER}' if DATA_SOURCE == "cms-open" else '')
      + (f'  |  cached at: {S3_CURATED_URI}encounters.parquet' if S3_CURATED_URI else ''))
print(f'Admission window : {encounters.admission_date.min().date()}  →  {encounters.admission_date.max().date()}')
print('\nFirst 5 rows:')
encounters.head(5)


## 3. Attach 30-day readmission labels + recompute prior-utilization features

In [ ]:
labeled = attach_labels_and_priors(encounters)
rate = labeled[LABEL_COL].mean()
print(f'30-day readmission rate: {rate:.4f}  ({labeled[LABEL_COL].sum():,} positives / {len(labeled):,} rows)')
labeled[['beneficiary_id','admission_date','discharge_date','length_of_stay',
         'primary_diagnosis','prior_inpatient_90d','readmitted_30d']].head()

## 4. Patient-grouped 40 / 30 / 30 split

In [ ]:
train_df, test_df, val_df = split_train_test_val(labeled, seed=42)

summary = pd.DataFrame({
    'fold':       ['train','test','val'],
    'rows':       [len(train_df), len(test_df), len(val_df)],
    'patients':   [train_df.beneficiary_id.nunique(), test_df.beneficiary_id.nunique(),
                   val_df.beneficiary_id.nunique()],
    'pos_rate':   [train_df[LABEL_COL].mean(), test_df[LABEL_COL].mean(), val_df[LABEL_COL].mean()],
})
summary['row_pct']     = summary['rows']     / summary['rows'].sum()
summary['patient_pct'] = summary['patients'] / summary['patients'].sum()
summary

## 5. Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labeled.groupby('primary_diagnosis')[LABEL_COL].mean().sort_values().plot(
    kind='barh', ax=axes[0], title='Readmission rate by diagnosis chapter')
labeled['length_of_stay'].hist(bins=30, ax=axes[1])
axes[1].set_title('Length of stay distribution'); axes[1].set_xlabel('days')
plt.tight_layout(); plt.show()

## 5b. Feature engineering

The same `add_derived_columns` + `winsorize_utilization` + `build_preprocessor` pipeline that `src/readmit/models/train.py` invokes during SageMaker training. Running it here lets us inspect the engineered columns *before* model training and gives us the curated frames (`train_prep`, `val_prep`, `test_prep`) that feed Feature Store ingestion, baselining, and the training cells below.


In [ ]:
# Engineer the curated feature frames once and reuse them across training,
# evaluation, Feature Store ingestion, and Model Monitor baselining.
def engineer(df):
    """Apply the production feature pipeline and return (X, y, frame_with_features)."""
    fe = add_derived_columns(winsorize_utilization(df))
    return fe[FEATURES.all], fe[LABEL_COL].to_numpy(), fe

x_train, y_train, train_prep = engineer(train_df)
x_val,   y_val,   val_prep   = engineer(val_df)
x_test,  y_test,  test_prep  = engineer(test_df)

# Inspect the preprocessor that will be wrapped inside every sklearn pipeline.
preprocessor = build_preprocessor(scale_numeric=True)
print('Numeric features    :', FEATURES.numeric)
print('Categorical features:', FEATURES.categorical)
print('\nEngineered columns on train_prep:')
train_prep[FEATURES.all].head()


## 5c. Feature Store registration

Register the curated training features in **Amazon SageMaker Feature Store** so the offline store (S3 + Glue table) backs training-set assembly and the online store (low-latency lookup keyed by `beneficiary_id`) backs real-time inference. Both stores share one schema, which is what stops train/serve skew.


In [ ]:
from readmit.features.feature_store import (
    create_or_update_feature_group,
    get_latest_features,
    ingest_features,
    prepare_records,
)

# Source frame for ingestion: keep the record id, the engineered features, the
# label (handy for offline back-tests), and the event-time anchor.
fs_cols = ['beneficiary_id', 'discharge_date', LABEL_COL] + FEATURES.all
fs_frame = train_prep[fs_cols].copy()
fs_records = prepare_records(fs_frame)
fs_records.head()

In [ ]:
if SAGEMAKER_ROLE_ARN:
    fg = create_or_update_feature_group(
        fs_records,
        role_arn=SAGEMAKER_ROLE_ARN,
        name=FEATURE_GROUP_NAME,
        s3_uri_prefix=f's3://{BUCKET}/{PROJECT_PREFIX}/feature-store',
        region=AWS_REGION,
        enable_online_store=True,
        description='30-day readmission features (AAI-540 Group 2 / ClearPath Health Analytics).',
    )
    ingest_features(fs_records, name=FEATURE_GROUP_NAME, region=AWS_REGION)
    # Read-back smoke test against the online store
    sample_id = str(fs_records['beneficiary_id'].iloc[0])
    online_row = get_latest_features([sample_id], name=FEATURE_GROUP_NAME, region=AWS_REGION)
    print('Online-store read-back for', sample_id)
    online_row
else:
    print('Skipping Feature Store registration — SAGEMAKER_ROLE_ARN is not set.')


## 5d. Offline-store training-set assembly via Athena

Now that the FeatureGroup is registered, the offline store is queryable as a Glue-cataloged Athena table — same data the SageMaker Pipeline (Section 9) uses to assemble training sets. This is the rubric's *offline-store* deliverable: ad-hoc analytical SQL over the curated features, joined to the label, with the guarantee that the same row a model trained on can be retrieved from the online store at inference time.

We do three things here:

1. Resolve the auto-generated Athena table name from the FeatureGroup.
2. Pull the engineered features back as a DataFrame (just like a Pipeline ProcessingStep would).
3. Run one analytical query — readmission rate by `age_band` × `primary_dx_chapter` — to show offline-store usability.


In [ ]:
if SAGEMAKER_ROLE_ARN:
    from readmit.features.feature_store import query_offline_features
    from sagemaker.feature_store.feature_group import FeatureGroup
    from sagemaker.session import Session as _SmSession
    import boto3 as _boto3

    # ----- 1. Resolve the Athena table name auto-created by Feature Store ------
    _sm_sess  = _SmSession(boto_session=_boto3.Session(region_name=AWS_REGION))
    _fg       = FeatureGroup(name=FEATURE_GROUP_NAME, sagemaker_session=_sm_sess)
    _athena   = _fg.athena_query()
    OFFLINE_DB    = _athena.database
    OFFLINE_TABLE = _athena.table_name
    ATHENA_OUTPUT = f's3://{BUCKET}/{PROJECT_PREFIX}/athena-results/'
    print(f'Offline-store Athena table: "{OFFLINE_DB}"."{OFFLINE_TABLE}"')
    print(f'Athena results staged at  : {ATHENA_OUTPUT}')

    # ----- 2. Pull a training-set slice back through Athena --------------------
    # Pipeline ProcessingStep would issue a query like this to build train.csv.
    # We grab a 5 000-row sample to keep the cell fast and free of huge scans.
    train_assembly_sql = f'''
        SELECT beneficiary_id, {", ".join(FEATURES.all)}, {LABEL_COL}
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        LIMIT 5000
    '''
    offline_train = query_offline_features(
        query=train_assembly_sql,
        name=FEATURE_GROUP_NAME,
        region=AWS_REGION,
        output_location=ATHENA_OUTPUT,
    )
    print(f'\nAthena returned {len(offline_train):,} rows × {offline_train.shape[1]} columns')
    offline_train.head(5)
else:
    print('Skipping Athena offline-store query — SAGEMAKER_ROLE_ARN is not set.')


In [ ]:
if SAGEMAKER_ROLE_ARN:
    # ----- 3. One analytical query that exercises Athena predicate push-down -
    # Readmission rate by age band x primary diagnosis chapter — the kind of
    # cohort question a care-ops analyst would ask without re-running the
    # training pipeline.
    cohort_sql = f'''
        SELECT
            age_band,
            primary_dx_chapter,
            COUNT(*)                                AS n_encounters,
            SUM({LABEL_COL})                        AS n_readmits,
            ROUND(AVG(CAST({LABEL_COL} AS DOUBLE)), 4) AS readmit_rate
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        GROUP BY age_band, primary_dx_chapter
        ORDER BY age_band, readmit_rate DESC
    '''
    cohort = query_offline_features(
        query=cohort_sql,
        name=FEATURE_GROUP_NAME,
        region=AWS_REGION,
        output_location=ATHENA_OUTPUT,
    )
    print(f'Cohort breakdown — {len(cohort)} (age_band x primary_dx_chapter) cells')
    cohort
else:
    print('Skipping Athena cohort query — SAGEMAKER_ROLE_ARN is not set.')


### 5d.1 Analytical queries against the offline store

Five queries that exercise the offline store the way different stakeholders would —
each one is a question we could answer **without retraining or touching the online endpoint**.
The point is to demonstrate that the feature store is a *governed, queryable system of record*,
not just a training-data dump.

| # | Audience              | Question                                                                  |
|---|-----------------------|---------------------------------------------------------------------------|
| 1 | Care-ops              | How is risk distributed today? (decile distribution + observed lift)      |
| 2 | Clinical leadership   | Which age × sex × prior-utilization cells drive the most readmits?        |
| 3 | Data engineering      | Coverage / data-quality across feature columns (null and zero rates)      |
| 4 | ML / MLOps            | How is the feature store growing over time? (ingest cadence by month)     |
| 5 | Targeting analyst     | Top-100 highest-risk recent encounters — a "today's worklist" preview     |

In [ ]:
if SAGEMAKER_ROLE_ARN:
    # ----- Query 1: risk-decile distribution --------------------------------
    # Bucket the readmission label by patient age band to show how baseline
    # risk varies. NTILE-style deciles aren't useful on a 0/1 label, so we
    # use age_band as the natural decile here.
    q1_sql = f'''
        SELECT
            age_band,
            COUNT(*)                                              AS n_encounters,
            SUM({LABEL_COL})                                      AS n_readmits,
            ROUND(AVG(CAST({LABEL_COL} AS DOUBLE)), 4)            AS readmit_rate,
            ROUND(AVG(CAST(prior_admits_180d AS DOUBLE)), 2)      AS avg_prior_admits_180d,
            ROUND(AVG(CAST(los_days AS DOUBLE)), 2)               AS avg_los_days
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        GROUP BY age_band
        ORDER BY age_band
    '''
    q1 = query_offline_features(
        query=q1_sql, name=FEATURE_GROUP_NAME,
        region=AWS_REGION, output_location=ATHENA_OUTPUT,
    )
    print(f'Q1 — Risk distribution by age band ({len(q1)} rows)')
    display(q1)
else:
    print('Skipping Q1 — SAGEMAKER_ROLE_ARN is not set.')


In [ ]:
if SAGEMAKER_ROLE_ARN:
    # ----- Query 2: clinical leadership cube --------------------------------
    # Three-way breakdown (age x sex x prior-utilization tier). Surface only
    # the most populated cells so the result fits on screen and the rates are
    # statistically meaningful.
    # NOTE: `CUBE` is a reserved keyword in Athena/Trino, so the CTE is named
    # `risk_cells` (not `cube`) to avoid an `InvalidRequestException`.
    q2_sql = f'''
        WITH risk_cells AS (
            SELECT
                age_band,
                sex,
                CASE
                    WHEN prior_admits_180d = 0 THEN 'no_prior'
                    WHEN prior_admits_180d = 1 THEN '1_prior'
                    WHEN prior_admits_180d BETWEEN 2 AND 3 THEN '2-3_prior'
                    ELSE '4+_prior'
                END                                               AS prior_admits_tier,
                COUNT(*)                                          AS n_encounters,
                SUM({LABEL_COL})                                  AS n_readmits,
                AVG(CAST({LABEL_COL} AS DOUBLE))                  AS readmit_rate
            FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
            WHERE is_deleted = false
            GROUP BY 1, 2, 3
        )
        SELECT
            age_band, sex, prior_admits_tier,
            n_encounters, n_readmits,
            ROUND(readmit_rate, 4)                                AS readmit_rate
        FROM risk_cells
        WHERE n_encounters >= 25                                  -- ignore tiny cells
        ORDER BY readmit_rate DESC
        LIMIT 25
    '''
    q2 = query_offline_features(
        query=q2_sql, name=FEATURE_GROUP_NAME,
        region=AWS_REGION, output_location=ATHENA_OUTPUT,
    )
    print(f'Q2 — Top-25 highest-risk (age x sex x prior-utilization) cells')
    display(q2)
else:
    print('Skipping Q2 — SAGEMAKER_ROLE_ARN is not set.')


In [ ]:
if SAGEMAKER_ROLE_ARN:
    # ----- Query 3: data-quality scan ---------------------------------------
    # For every numeric feature we engineered, surface coverage stats so a
    # data engineer can spot drift / pipeline regressions at a glance.
    numeric_feats = [c for c in FEATURES.numeric if c != LABEL_COL]
    union_parts = []
    for col in numeric_feats:
        union_parts.append(f"""
        SELECT
            '{col}'                                               AS feature,
            COUNT(*)                                              AS n_rows,
            SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END)        AS n_nulls,
            SUM(CASE WHEN {col} = 0 THEN 1 ELSE 0 END)            AS n_zeros,
            ROUND(MIN(CAST({col} AS DOUBLE)), 4)                  AS min_v,
            ROUND(AVG(CAST({col} AS DOUBLE)), 4)                  AS mean_v,
            ROUND(MAX(CAST({col} AS DOUBLE)), 4)                  AS max_v
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        """)
    q3_sql = "\nUNION ALL\n".join(union_parts) + "\nORDER BY feature"

    q3 = query_offline_features(
        query=q3_sql, name=FEATURE_GROUP_NAME,
        region=AWS_REGION, output_location=ATHENA_OUTPUT,
    )
    q3['null_pct'] = (q3['n_nulls'] / q3['n_rows']).round(4)
    q3['zero_pct'] = (q3['n_zeros'] / q3['n_rows']).round(4)
    print(f'Q3 — Coverage / quality across {len(q3)} engineered numeric features')
    display(q3[['feature', 'n_rows', 'null_pct', 'zero_pct', 'min_v', 'mean_v', 'max_v']])
else:
    print('Skipping Q3 — SAGEMAKER_ROLE_ARN is not set.')


In [ ]:
if SAGEMAKER_ROLE_ARN:
    # ----- Query 4: ingest cadence ------------------------------------------
    # `event_time` is stored as a float (seconds since epoch) per Feature
    # Store convention — bucket it back into months for a growth view.
    q4_sql = f'''
        SELECT
            DATE_FORMAT(
                FROM_UNIXTIME(CAST(event_time AS BIGINT)),
                '%Y-%m'
            )                                                     AS ingest_month,
            COUNT(*)                                              AS n_records,
            COUNT(DISTINCT {RECORD_ID_COL_SQL})                   AS n_distinct_patients,
            SUM({LABEL_COL})                                      AS n_readmits,
            ROUND(AVG(CAST({LABEL_COL} AS DOUBLE)), 4)            AS readmit_rate
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        GROUP BY 1
        ORDER BY ingest_month
    '''.replace('{RECORD_ID_COL_SQL}', 'beneficiary_id')

    q4 = query_offline_features(
        query=q4_sql, name=FEATURE_GROUP_NAME,
        region=AWS_REGION, output_location=ATHENA_OUTPUT,
    )
    print(f'Q4 — Feature-store growth: {len(q4)} months of ingest activity')
    display(q4)

    # Visual aid — small bar chart of records per month (when matplotlib is up).
    if len(q4) > 1:
        ax = q4.plot.bar(
            x='ingest_month', y='n_records',
            figsize=(10, 3), legend=False, color='#3b7dd8',
            title='Feature-store records ingested per encounter month',
        )
        ax.set_ylabel('records'); ax.set_xlabel('')
        plt.tight_layout(); plt.show()
else:
    print('Skipping Q4 — SAGEMAKER_ROLE_ARN is not set.')


In [ ]:
if SAGEMAKER_ROLE_ARN:
    # ----- Query 5: today's "worklist" preview ------------------------------
    # The most recent N encounters with the highest model-relevant risk
    # signals. This is the shape of query a care-ops UI would issue against
    # the offline store to backfill a daily intervention queue (the online
    # store would serve real-time lookups for individual patients).
    q5_sql = f'''
        SELECT
            beneficiary_id,
            FROM_UNIXTIME(CAST(event_time AS BIGINT))             AS event_ts,
            age_band, sex, primary_dx_chapter,
            prior_admits_180d, prior_ed_visits_180d, los_days,
            {LABEL_COL}                                           AS observed_readmit
        FROM "{OFFLINE_DB}"."{OFFLINE_TABLE}"
        WHERE is_deleted = false
        ORDER BY prior_admits_180d DESC,
                 prior_ed_visits_180d DESC,
                 los_days DESC,
                 event_time DESC
        LIMIT 100
    '''
    q5 = query_offline_features(
        query=q5_sql, name=FEATURE_GROUP_NAME,
        region=AWS_REGION, output_location=ATHENA_OUTPUT,
    )
    print(f'Q5 — Top-100 high-utilization encounters (worklist preview)')
    display(q5.head(20))
    print(f'... {len(q5) - 20} more rows in q5')
else:
    print('Skipping Q5 — SAGEMAKER_ROLE_ARN is not set.')


## 6a. Baseline — logistic regression

In [ ]:
class _Args:
    learning_rate=0.1; max_depth=4; n_estimators=400
    subsample=0.9; colsample_bytree=0.9; reg_lambda=1.0; seed=42

# x_train / y_train / *_prep already built in section 5b above.
lr_pipeline = build_logreg_pipeline(_Args()).fit(x_train, y_train)
lr_proba    = lr_pipeline.predict_proba(x_val)[:, 1]
lr_metrics  = compute_metrics(y_val, lr_proba)
print('Logistic-regression validation metrics:')
json.dumps(lr_metrics, indent=2)


## 6b. Primary model — XGBoost

In [ ]:
pos = int(y_train.sum()); neg = int(len(y_train) - pos)
scale_pos_weight = max(neg / max(pos, 1), 1.0)
print(f'Class imbalance: pos={pos}, neg={neg}, scale_pos_weight={scale_pos_weight:.3f}')

xgb_pipeline = build_xgb_pipeline(_Args(), scale_pos_weight=scale_pos_weight).fit(x_train, y_train)
xgb_proba    = xgb_pipeline.predict_proba(x_val)[:, 1]
xgb_metrics  = compute_metrics(y_val, xgb_proba)
xgb_metrics['recall_at_top_10pct'] = recall_at_k(y_val, xgb_proba, 0.10)
xgb_metrics['recall_at_top_20pct'] = recall_at_k(y_val, xgb_proba, 0.20)
json.dumps(xgb_metrics, indent=2)

## 7. Subgroup / fairness evaluation

In [ ]:
subgroup = subgroup_metrics(
    val_prep, y_val, xgb_proba,
    group_cols=['age_band','sex','primary_dx_chapter'],
)
subgroup[['group','value','n','n_pos','auc','pr_auc','precision','recall']]

## 8. Final test-set evaluation + deploy gate

The AUC gate (`AUC_THRESHOLD_DEPLOY = 0.75`) is the same threshold enforced by the SageMaker Pipeline `ConditionStep` and the CI/CD workflow.

In [ ]:
test_proba = xgb_pipeline.predict_proba(x_test)[:, 1]
test_metrics = compute_metrics(y_test, test_proba)
test_metrics['recall_at_top_10pct'] = recall_at_k(y_test, test_proba, 0.10)
test_metrics['recall_at_top_20pct'] = recall_at_k(y_test, test_proba, 0.20)
print(json.dumps(test_metrics, indent=2))

passed = test_metrics['auc'] >= AUC_THRESHOLD_DEPLOY
print(f"\nAUC gate (>= {AUC_THRESHOLD_DEPLOY}): {'PASS' if passed else 'FAIL — synthetic data may not meet target; real CMS data is expected to.'}")

## 8b. Model Store — package + register the trained model

Persist the locally-trained XGBoost pipeline as a SageMaker **ModelPackage** in the `ReadmitRiskModels` ModelPackageGroup. Versions enter as `PendingManualApproval`; the AUC-gate helper auto-approves only when the held-out test AUC clears `AUC_THRESHOLD_DEPLOY` (= 0.75). CD then deploys the most recent `Approved` version — never an un-gated one.


In [ ]:
if SAGEMAKER_ROLE_ARN:
    import joblib, tarfile, tempfile, pathlib
    from readmit.models.registry import (
        APPROVED,
        auto_approve_if_above_threshold,
        ensure_model_package_group,
        list_versions,
        register_model_version,
    )

    ensure_model_package_group(
        group_name=MODEL_PACKAGE_GROUP,
        region=AWS_REGION,
    )

    # 1. Bundle the trained sklearn Pipeline into model.tar.gz alongside the
    #    inference contract (src/readmit/models/inference.py is the entry point).
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = pathlib.Path(tmp)
        joblib.dump(xgb_pipeline, tmp_path / 'model.joblib')
        (tmp_path / 'feature_spec.json').write_text(json.dumps({
            'numeric': FEATURES.numeric,
            'categorical': FEATURES.categorical,
            'label': LABEL_COL,
        }, indent=2))
        (tmp_path / 'evaluation.json').write_text(json.dumps(test_metrics, indent=2))

        archive = tmp_path / 'model.tar.gz'
        with tarfile.open(archive, 'w:gz') as tar:
            for name in ('model.joblib', 'feature_spec.json', 'evaluation.json'):
                tar.add(tmp_path / name, arcname=name)

        # 2. Upload artifact + metrics to S3
        model_s3 = sess.upload_data(
            str(archive), bucket=BUCKET,
            key_prefix=f'{PROJECT_PREFIX}/model-store/artifacts',
        )
        metrics_s3 = sess.upload_data(
            str(tmp_path / 'evaluation.json'), bucket=BUCKET,
            key_prefix=f'{PROJECT_PREFIX}/model-store/metrics',
        )

    # 3. Register the version (always starts as PendingManualApproval)
    pkg_arn = register_model_version(
        model_data_s3_uri=model_s3,
        group_name=MODEL_PACKAGE_GROUP,
        region=AWS_REGION,
        metrics_s3_uri=metrics_s3,
        description=f"Notebook-trained XGBoost — test AUC {test_metrics['auc']:.4f}",
        customer_metadata={
            'trained_in': 'notebook',
            'train_rows': len(x_train),
            'split_ratio': '40/30/30',
            'auc_threshold_deploy': AUC_THRESHOLD_DEPLOY,
        },
    )
    print('Registered ModelPackageArn:', pkg_arn)

    # 4. Auto-approve only if test AUC clears the gate
    approved = auto_approve_if_above_threshold(
        pkg_arn, test_metrics['auc'],
        threshold=AUC_THRESHOLD_DEPLOY, region=AWS_REGION,
    )
    print('Auto-approval result      :', 'APPROVED' if approved else 'REJECTED')

    # 5. Show the registry contents
    versions = list_versions(group_name=MODEL_PACKAGE_GROUP, region=AWS_REGION)
    pd.DataFrame([{
        'arn'    : v['ModelPackageArn'].split('/')[-1],
        'status' : v['ModelApprovalStatus'],
        'created': v['CreationTime'],
    } for v in versions[:10]])
else:
    print('Skipping Model Store registration — SAGEMAKER_ROLE_ARN is not set.')


## 9. (Optional) Run the production SageMaker Pipeline

Skip this section if you only want to train locally. Requires `SAGEMAKER_ROLE_ARN` to be set.

In [ ]:
if SAGEMAKER_ROLE_ARN:
    from readmit.pipeline.sagemaker_pipeline import build_pipeline

    pipeline = build_pipeline(
        role_arn=SAGEMAKER_ROLE_ARN,
        pipeline_name='ReadmitRiskPipeline',
        model_package_group='ReadmitRiskModels',
        data_source=DATA_SOURCE,
        s3_data_uri=S3_CURATED_URI,
        n_patients=N_PATIENTS,
        region=AWS_REGION,
        auc_threshold=AUC_THRESHOLD_DEPLOY,
    )
    pipeline.upsert(role_arn=SAGEMAKER_ROLE_ARN)
    execution = pipeline.start()
    print('Pipeline started:', execution.arn)
    # execution.wait()   # uncomment to block the notebook until pipeline completes
else:
    print('Skipping SageMaker Pipeline run — SAGEMAKER_ROLE_ARN is not set.')

## 10. Deploy a real-time endpoint (notebook-driven path)

For the classroom demo we deploy the locally-trained XGBoost pipeline as a SageMaker model + real-time endpoint with **data capture enabled**, so Model Monitor has traffic to evaluate.

In [ ]:
if SAGEMAKER_ROLE_ARN:
    import joblib, tarfile, tempfile
    import boto3
    from sagemaker.session import Session
    from sagemaker.sklearn.model import SKLearnModel
    from sagemaker.model_monitor import DataCaptureConfig

    sess = Session(boto_session=boto3.Session(region_name=AWS_REGION))

    # 1. Pack the trained pipeline as model.tar.gz and upload to S3.
    with tempfile.TemporaryDirectory() as tmp:
        joblib.dump(xgb_pipeline, os.path.join(tmp, 'model.joblib'))
        tar_path = os.path.join(tmp, 'model.tar.gz')
        with tarfile.open(tar_path, 'w:gz') as t:
            t.add(os.path.join(tmp, 'model.joblib'), arcname='model.joblib')
        s3_model_uri = sess.upload_data(tar_path, bucket=BUCKET,
                                        key_prefix=f'{PROJECT_PREFIX}/models')
    print('Uploaded model artifact to', s3_model_uri)

    # 2. Wrap it in an SKLearnModel pointing at our inference handler.
    capture_uri = f's3://{BUCKET}/{PROJECT_PREFIX}/data-capture'
    sk_model = SKLearnModel(
        model_data=s3_model_uri,
        role=SAGEMAKER_ROLE_ARN,
        entry_point='readmit/models/inference.py',
        source_dir='../src',
        framework_version='1.2-1', py_version='py3',
        sagemaker_session=sess,
    )

    # 3. Deploy with data capture for Model Monitor.
    predictor = sk_model.deploy(
        initial_instance_count=1,
        instance_type='ml.m5.large',
        endpoint_name=ENDPOINT_NAME,
        data_capture_config=DataCaptureConfig(
            enable_capture=True, sampling_percentage=100,
            destination_s3_uri=capture_uri,
            sagemaker_session=sess,
        ),
    )
    print('Endpoint deployed:', ENDPOINT_NAME)
else:
    print('Skipping endpoint deploy — SAGEMAKER_ROLE_ARN is not set.')

## 11. Model Monitor — baseline + hourly drift schedule

Builds a baseline statistics + constraints set from the training data and schedules an hourly drift check against captured inference traffic.

In [ ]:
if SAGEMAKER_ROLE_ARN:
    from readmit.monitoring.setup_model_monitor import (
        create_data_quality_baseline, create_data_quality_schedule,
    )

    # Upload the training feature matrix as the monitor baseline.
    baseline_local = '/tmp/monitor_baseline.csv'
    train_prep[FEATURES.all].to_csv(baseline_local, index=False)
    baseline_s3 = sess.upload_data(
        baseline_local, bucket=BUCKET,
        key_prefix=f'{PROJECT_PREFIX}/monitor/baseline-input',
    )
    baseline_out = f's3://{BUCKET}/{PROJECT_PREFIX}/monitor/baseline-output'
    monitor = create_data_quality_baseline(
        role_arn=SAGEMAKER_ROLE_ARN,
        baseline_dataset_s3_uri=baseline_s3,
        output_s3_uri=baseline_out,
        region=AWS_REGION,
    )
    create_data_quality_schedule(
        monitor, endpoint_name=ENDPOINT_NAME,
        output_s3_uri=f's3://{BUCKET}/{PROJECT_PREFIX}/monitor/reports',
    )
    print('Model Monitor schedule active for', ENDPOINT_NAME)
else:
    print('Skipping Model Monitor setup — SAGEMAKER_ROLE_ARN is not set.')

## 12. CloudWatch alarms + SNS alerting


In [ ]:
from readmit.monitoring.alerts import build_dashboard, configure_alarms

topic_arn = configure_alarms(
    endpoint_name=ENDPOINT_NAME,
    region=AWS_REGION,
)
dashboard_name = build_dashboard(
    endpoint_name=ENDPOINT_NAME,
    region=AWS_REGION,
)
print('Alarm topic ARN :', topic_arn)
print('Dashboard       :', dashboard_name)
print(f'Dashboard URL   : https://{AWS_REGION}.console.aws.amazon.com/cloudwatch/home?region={AWS_REGION}#dashboards:name={dashboard_name}')


## 13. Live inference test

In [ ]:
if SAGEMAKER_ROLE_ARN:
    import boto3
    smr = boto3.client('sagemaker-runtime', region_name=AWS_REGION)
    payload = {'instances': [{
        'age': 72, 'sex': 'F', 'primary_diagnosis': 'circulatory',
        'discharge_disposition': 'home', 'payer_type': 'medicare_ffs',
        'length_of_stay': 5, 'n_chronic_conditions': 4, 'charlson_index': 3,
        'prior_inpatient_90d': 1, 'prior_ed_90d': 2, 'prior_outpatient_90d': 3,
    }]}
    resp = smr.invoke_endpoint(
        EndpointName=ENDPOINT_NAME, ContentType='application/json',
        Accept='application/json', Body=json.dumps(payload),
    )
    print(json.loads(resp['Body'].read()))
else:
    # Local invocation goes through the same code path the endpoint serves.
    from readmit.models.inference import input_fn, predict_fn, output_fn
    payload = '{"instances": [{"age":72,"sex":"F","primary_diagnosis":"circulatory","discharge_disposition":"home","payer_type":"medicare_ffs","length_of_stay":5,"n_chronic_conditions":4,"charlson_index":3,"prior_inpatient_90d":1,"prior_ed_90d":2,"prior_outpatient_90d":3}]}'
    x   = input_fn(payload, 'application/json')
    out, _ = output_fn(predict_fn(x, xgb_pipeline), 'application/json')
    print(out)

## 14. Cleanup

Only run this when you are done — it deletes the endpoint to stop billing.

In [ ]:
# import boto3
# sm = boto3.client('sagemaker', region_name=AWS_REGION)
# sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
# sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
# print('Endpoint deleted')

## 15. Model Card

A short, rubric-friendly summary of what this model is, what it's for, and
where it should *not* be used. Mirrors the Google / HuggingFace model-card
template.

### Model details
- **Name:** `ReadmitRiskModels` (primary version: XGBoost; baseline: logistic regression)
- **Owners:** AAI-540 Group 2 — Jose Sandoval, Manikanta Katuri, Michael Domingo
- **Version:** auto-incremented in the SageMaker Model Registry; only versions
  with held-out test AUC ≥ `AUC_THRESHOLD_DEPLOY` (= 0.75) are auto-approved.
- **License / data use:** the training data is **CMS DE-SynPUF** — a
  publicly-released, **synthesised** sample of 2008–2010 Medicare FFS claims
  that CMS generated from real claims and de-identified for unrestricted
  research / educational use. It contains **no PHI**. This model inherits
  that license.

### Intended use
- **Primary use case:** rank inpatient discharges by 30-day all-cause
  readmission risk so a care-management team can prioritise post-discharge
  outreach (med reconciliation calls, transitional-care visits).
- **Intended users:** care-coordination analysts and clinical informatics
  teams inside a Medicare-focused payer or ACO.
- **Out-of-scope uses:** ❌ individual treatment decisions, ❌ admission /
  discharge denials, ❌ insurance pricing, ❌ any use on pediatric, non-Medicare,
  or non-US populations.

### Training data
- **Source:** CMS DE-SynPUF (Data Entrepreneurs' **Syn**thetic Public Use File)
  in OMOP CDM v5.x, hosted on the AWS Open Data Registry at
  `s3://synpuf-omop/` — 100k-beneficiary tier.
- **Provenance:** CMS produced DE-SynPUF by sampling 5 % of 2008–2010
  Medicare FFS claims and applying a structured synthesis / perturbation
  process so the released records preserve statistical properties (age,
  diagnosis, utilisation distributions) without corresponding to any real
  beneficiary. Switching to real CMS claims (the RIF / LDS files) only
  requires changing `DATA_SOURCE` to point at a private bucket with a signed
  Data Use Agreement; the pipeline code is identical.
- **Window:** claims dated 2008–2010 (the full DE-SynPUF release).
- **Cohort:** adult Medicare beneficiaries with at least one inpatient
  encounter; `length_of_stay` clipped to 1–30 days.
- **Label:** `readmitted_30d = 1` iff a subsequent inpatient admission for the
  same beneficiary occurs within 30 days of the index discharge.
- **Split:** 40 / 30 / 30 train / test / validation, **patient-grouped** and
  time-aware (see `src/readmit/data/splits.py`).

### Features
14 columns: demographics (`age`, `sex`), encounter (`length_of_stay`,
`primary_diagnosis`, `discharge_disposition`, `payer_type`), comorbidity
burden (`n_chronic_conditions`, `charlson_index`), and 90-day prior
utilisation (`prior_inpatient_90d`, `prior_ed_90d`, `prior_outpatient_90d`).
Feature definitions and the engineering pipeline are versioned in
`src/readmit/features/` and registered in **SageMaker Feature Store**
(group `readmit-encounter-features`).

### Metrics & evaluation
- **Primary:** AUC-ROC (deploy gate ≥ 0.75)
- **Secondary:** PR-AUC, Recall@top-10%, Recall@top-20%, Brier score
- **Fairness:** per-subgroup AUC / PR-AUC / precision / recall across
  `age_band`, `sex`, and `primary_dx_chapter` (Section 7)
- Last-run numbers are persisted next to the model artifact in the registry
  as `evaluation.json`.

### Known limitations
1. **DE-SynPUF is a synthesised sample, not live claims.** It is statistically
   representative of 2008–2010 Medicare FFS but is not a real patient
   population. Real-world AUC is expected to be in the same ballpark but is
   unverified until the model is retrained on actual CMS RIF / LDS data.
2. **Payer mix is heuristic.** DE-SynPUF doesn't expose payer detail in the
   1k / 100k tiers, so `payer_type` is derived deterministically from
   `person_id` (≈ 55 % FFS / 30 % MA / 15 % dual). This will not match any
   real plan's distribution and should be replaced with the payer file on
   real claims.
3. **Diagnosis chapters are bucketed.** Primary diagnosis is hashed into 8
   stable chapters because the OMOP concept dictionary isn't shipped with
   the open dataset. A production deployment should swap this for an
   ICD-10 → AHRQ-CCS mapping.
4. **No social-determinants features.** Income, language, transportation,
   and prior medication adherence are not in DE-SynPUF; they materially
   affect real readmission risk and would change subgroup behaviour.
5. **Distribution shift.** The 2008–2010 training window predates COVID-era
   utilisation patterns; expect drift on contemporary data and rely on
   Model Monitor (Section 11) to surface it.

### Ethical / clinical considerations
- The model surfaces a *risk score*, not a diagnosis. Care teams retain full
  clinical judgement.
- Subgroup metrics are reported in Section 7 so reviewers can check that
  performance does not collapse for any demographic.
- Because DE-SynPUF is de-identified by construction, no consent, BAA, or
  IRB review is required for this prototype. A production version on real
  CMS RIF / LDS claims would require a signed DUA, encrypted-at-rest storage,
  IAM-scoped access, and an IRB-style review for clinical deployment.

### Maintenance / retraining cadence
- **Scheduled retrain:** monthly via the CD workflow (`.github/workflows/cd.yml`)
  running the SageMaker Pipeline.
- **Drift-triggered retrain:** Model Monitor publishes a custom CloudWatch
  metric; the `ReadmitMonitor-DataQualityViolations` alarm (Section 12)
  triggers an ad-hoc pipeline run.
- **Rollback:** previous `Approved` versions stay in the Model Registry;
  the CD job can promote any prior version by ARN.
- **Contact:** open an issue in the GitHub repo; tag `@aai540-group2`.
